# AudioTrain: Neural Amp Modeler & Hardware Exporter (Colab Edition)
### Google Colab Training Suite for SincoANN Hardware (AM3 / AM4 / IR / NAM Distillation)

This notebook trains neural amplifier models from:
- **2-Channel Stereo Audio WAVs**: Channel 0 = Dry input, Channel 1 = Amp/Pedal output.
- **Neural Amp Modeler (.nam) Profiles**: Automatic distillation supporting both **A1 WaveNet** and modern **A2 Architecture** (`A2-Full` / `A2-Lite` / `SlimmableContainer`).

**Hardware Output:** Byte-exact `.am4Data`, `.am3Data`, and `.irData` profiles ready to load directly into Sinco hardware and plugins.

## 1. Setup Environment & GPU Check
Ensure you are connected to a **T4 GPU** runtime (`Runtime` -> `Change runtime type` -> `T4 GPU`).

In [ ]:
# Check GPU availability
!nvidia-smi

import os
# If you uploaded the standalone audiotrain_colab.zip, extract it:
if os.path.exists("audiotrain_colab.zip"):
    !unzip -qo audiotrain_colab.zip

# Install dependencies and audiotrain in editable mode
!pip install -q soundfile pyyaml scipy
!pip install -q -e .

import torch
import audiotrain
print(f"AudioTrain v{audiotrain.__version__} ready!")
print(f"CUDA Available: {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

## 2. Provide Training Audio or NAM Profiles
Upload one or more stereo WAV files or `.nam` models into `samples/`, or mount Google Drive.
- **WAV files**: 2-channel stereo (dry on left, wet target on right)
- **NAM files**: Any `.nam` model (A1 or A2). AudioTrain automatically uses its bundled 44.1 kHz dry excitation audio track to synthesize targets!

In [ ]:
import os
from google.colab import files

os.makedirs("samples", exist_ok=True)
print("Upload your stereo WAV training file(s) or .nam model(s) below (or skip if mounting Drive):")
uploaded = files.upload()

for fn in uploaded.keys():
    dest = os.path.join("samples", fn)
    os.rename(fn, dest)
    print(f"Saved: {dest}")

# Option B: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# SAMPLES_SOURCE = '/content/drive/MyDrive/MyAmps/'

## 3. Configure & Run Training
Select your target architecture:
- `AM4`: 512-tap FIR -> GRU(16) -> Linear -> 512-tap FIR (Standard Sinco Profile, 1,952 params)
- `AM3`: CNN -> 256-tap FIR -> asinh -> GRU(16) -> Linear -> 256-tap FIR (1,536 params)
- `IR1024` / `IR2048`: Linear impulse response convolution
- `NAM_TO_AM4` / `NAM_TO_AM3`: Distill a Neural Amp Modeler (.nam) profile

For **NAM A2** models, `--nam-variant full` distills from the 8-channel high-fidelity tier (default), while `--nam-variant lite` distills from the 3-channel lightweight tier.

In [ ]:
# Target: Can be a single file ('samples/clean.wav') OR an entire folder ('samples/') for batch queue
SAMPLES_SOURCE = "samples/"  
MODEL_TYPE = "AM4"          # Options: AM4, AM3, IR1024, IR2048, NAM_TO_AM4, NAM_TO_AM3
NAM_VARIANT = "full"        # For A2 NAM models: 'full' (8-channel default) or 'lite' (3-channel)
EPOCHS = 1000               # Full training epochs (~1-2 minutes per model on T4 GPU)
BATCH_SIZE = 1024           # 512-sample frames per batch
LEARNING_RATE = 0.0008      # AdamW learning rate
OUTPUT_DIR = "output"       # Directory for exported profiles
RENDER_AUDIO = False        # Set True to generate audio preview WAV (off by default for speed)
SHOW_PROGRESS = False       # Set True for granular batch-by-batch progress output

cmd = (
    f"python -m audiotrain.cli "
    f"--samples '{SAMPLES_SOURCE}' "
    f"--model {MODEL_TYPE} "
    f"--nam-variant {NAM_VARIANT} "
    f"--epochs {EPOCHS} "
    f"--batch-size {BATCH_SIZE} "
    f"--lr {LEARNING_RATE} "
    f"--output-dir '{OUTPUT_DIR}' "
    f"--device cuda "
    f"--skip-existing"
)

if RENDER_AUDIO:
    cmd += " --render-audio"
if SHOW_PROGRESS:
    cmd += " --show-progress"

!{cmd}

## 4. Listen to Audio Results in Notebook (Optional)
If `RENDER_AUDIO = True` was enabled, you can listen to the Dry input and the model's rendered Amp output side-by-side.

In [ ]:
import glob
import soundfile as sf
import IPython.display as ipd

wav_files = glob.glob(f"{OUTPUT_DIR}/*_am*.wav") + glob.glob(f"{OUTPUT_DIR}/*_ir.wav")
if wav_files:
    for out_wav in wav_files[:3]:  # Preview up to 3 models
        print(f"\n=== Audio Preview for: {out_wav} ===")
        data, sr = sf.read(out_wav)
        dry_signal = data[:, 0]
        amp_output = data[:, 1]
        
        print("Dry Guitar Input:")
        ipd.display(ipd.Audio(dry_signal[:sr*10], rate=sr))
        
        print("Trained Neural Amp Output:")
        ipd.display(ipd.Audio(amp_output[:sr*10], rate=sr))
else:
    print("No preview WAV found. To generate audio preview files, set RENDER_AUDIO = True in cell 3.")

## 5. Download Hardware Binary Profiles (`.am4Data` / `.am3Data`)
Download the finished profiles or zip the entire `output/` folder.

In [ ]:
import os
from google.colab import files

# Zip and download the entire output folder (recommended for batch runs)
zip_name = "output_profiles.zip"
!zip -q -r {zip_name} {OUTPUT_DIR}/
if os.path.exists(zip_name):
    print(f"Downloading {zip_name}...")
    files.download(zip_name)
else:
    print(f"No files found in {OUTPUT_DIR}")